# Dry-Run Validasi MoE (`Qwen/Qwen3-1.7B` -> `Qwen3 MoE`) -> GGUF di Colab T4

Notebook ini menguji kompatibilitas `mergekit-moe` (`gate_mode: cheap_embed`, `lazy_unpickle: true`) terhadap `Qwen/Qwen3-1.7B` dan konversi ke **GGUF `Q4_K_M`** sebelum melatih 4 expert secara penuh.

**Plan B (Jika konversi MoE -> GGUF bermasalah):** Gunakan 1 base model `Qwen2.5-3B-Instruct` + 4 Dynamic LoRA Adapters via `llama.cpp --lora`.

In [ ]:
!nvidia-smi
!pip install -q mergekit transformers accelerate safetensors sentencepiece
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -B /content/llama.cpp/build /content/llama.cpp && cmake --build /content/llama.cpp/build --target llama-quantize llama-cli -j4

In [ ]:
import yaml
from huggingface_hub import snapshot_download

base_dir = "/content/qwen3_1_7b_base"
snapshot_download("Qwen/Qwen3-1.7B", local_dir=base_dir)

dryrun_cfg = {
    "base_model": base_dir,
    "gate_mode": "cheap_embed",
    "lazy_unpickle": True,
    "dtype": "float16",
    "experts_per_token": 2,
    "experts": [
        {"source_model": base_dir, "positive_prompts": ["Buat jurnal akuntansi dan neraca lajur."]},
        {"source_model": base_dir, "positive_prompts": ["Hitung EBITDA dan kertas kerja audit."]},
    ],
}
with open("/content/dryrun_moe.yaml", "w") as f:
    yaml.safe_dump(dryrun_cfg, f)
print("Dry-run Qwen3-1.7B YAML ready.")

In [ ]:
!mergekit-moe /content/dryrun_moe.yaml /content/dryrun_moe_out --copy-tokenizer --allow-crimes --i-understand-this-is-not-useful-without-training && \
python3 /content/llama.cpp/convert_hf_to_gguf.py /content/dryrun_moe_out --outfile /content/dryrun_moe_f16.gguf --outtype f16 && \
/content/llama.cpp/build/bin/llama-quantize /content/dryrun_moe_f16.gguf /content/dryrun_moe_q4_k_m.gguf Q4_K_M && \
/content/llama.cpp/build/bin/llama-cli -m /content/dryrun_moe_q4_k_m.gguf -p "Hitung PPh 23 jasa audit Rp 10.000.000:" -n 64